# NPS Rolling Export: Delta Table → CSV → SharePoint

**Purpose**: Read the aggregated NPS Delta table written by the upstream Dataflow Gen2,
export it as a CSV file, and upload it to SharePoint via the Microsoft Graph API.

**Inputs**
- Fabric Lakehouse Delta table: `cx_nps_rolling_5w` (written by the Dataflow)

**Outputs**
- Timestamped CSV: `nps_rolling_5w_YYYYMMDD.csv` (daily archive)
- Latest snapshot: `nps_rolling_5w_LATEST.csv` (stable filename for consumers)
- Both files uploaded to the configured SharePoint folder

**Trigger context**: Runs as Activity 2 in the Fabric Pipeline, after the Dataflow
completes successfully. Should not be run in isolation unless the Delta table is
already up to date.

## Dependencies

| Library | Purpose |
|---|---|
| `pyspark` | Read the Lakehouse Delta table via the Fabric Spark session |
| `pandas` | Convert the Spark DataFrame and serialize to CSV |
| `requests` | HTTP calls to Microsoft Graph API (token + file upload) |
| `datetime` | Generate the date-stamped filename |
| `pathlib` | Create the local staging folder if it does not exist |

`pyspark` is provided by the Fabric runtime. All other libraries are available
in the default Fabric notebook environment.

In [ ]:
import os
import pandas as pd
import requests
from datetime import datetime
from pathlib import Path

from pyspark.sql import SparkSession

## Configuration

All sensitive values are read from environment variables. No secrets appear in this notebook.

In Microsoft Fabric, set these via:
- **Workspace-level environment**: Fabric workspace settings → Environment
- **Azure Key Vault**: Link a Key Vault to your Fabric workspace and reference secrets by name

For local testing, copy `.env.example` to `.env` and populate the values.
See [config/config_template.py](../config/config_template.py) for documentation
on where to find each value in the Azure Portal.

In [ ]:
spark = SparkSession.builder.getOrCreate()

# ── Fabric / Delta Table ──────────────────────────────────────────────────────
TABLE_NAME   = os.environ.get("NPS_TABLE_NAME", "cx_nps_rolling_5w")
LOCAL_FOLDER = os.environ.get("LOCAL_EXPORT_FOLDER", "/lakehouse/default/Files/dumps")

# ── Azure AD App Registration ─────────────────────────────────────────────────
TENANT_ID     = os.environ["AAD_TENANT_ID"]
CLIENT_ID     = os.environ["AAD_CLIENT_ID"]
CLIENT_SECRET = os.environ["AAD_CLIENT_SECRET"]

# ── SharePoint / Microsoft Graph ──────────────────────────────────────────────
SPO_SITE_PATH = os.environ["SPO_SITE_PATH"]
FOLDER_PATH   = os.environ["SPO_FOLDER_PATH"]
SITE_ID       = os.environ["SPO_SITE_ID"]
DRIVE_ID      = os.environ["SPO_DRIVE_ID"]

## Step 1: Read Delta Table

Read the aggregated NPS table from the Lakehouse using the active Spark session.
An explicit row-count check ensures the pipeline fails fast if the Dataflow
produced an empty result — preventing an empty CSV from silently overwriting
the previous good export in SharePoint.

In [ ]:
print(f"Reading Delta table: {TABLE_NAME}")

df_spark = spark.read.table(TABLE_NAME)
row_count = df_spark.count()

print(f"Rows read: {row_count}")

if row_count == 0:
    raise Exception(f"Table '{TABLE_NAME}' is empty. Aborting export to prevent overwriting the last good file.")

df = df_spark.toPandas()

print(f"Columns: {df.columns.tolist()}")
print(f"Shape: {df.shape}")

## Step 2: Export to CSV

Two files are written to the local Lakehouse staging folder:

1. **Timestamped archive** (`nps_rolling_5w_YYYYMMDD.csv`): one file per day,
   allowing point-in-time recovery and audit.
2. **Latest snapshot** (`nps_rolling_5w_LATEST.csv`): a stable filename that
   downstream consumers (e.g. a Power BI report) can reference without knowing
   the current date.

Both files are uploaded to SharePoint in the next step.

In [ ]:
print("Generating CSV files...")

today = datetime.today().strftime("%Y%m%d")
filename        = f"nps_rolling_5w_{today}.csv"
latest_filename = "nps_rolling_5w_LATEST.csv"

Path(LOCAL_FOLDER).mkdir(parents=True, exist_ok=True)

local_path  = f"{LOCAL_FOLDER}/{filename}"
latest_path = f"{LOCAL_FOLDER}/{latest_filename}"

df.to_csv(local_path,  index=False)
df.to_csv(latest_path, index=False)

print(f"Timestamped: {local_path}")
print(f"Latest:      {latest_path}")

## Step 3: Authenticate with Microsoft Graph

Authentication uses the **OAuth 2.0 client credentials flow** (also called
app-only authentication). This flow is appropriate for background services
and daemons that run without user interaction — the pipeline runs unattended
on a schedule, so delegated (user) authentication is not applicable.

The app registration in Azure AD must have the `Files.ReadWrite.All` (or
equivalent SharePoint) application permission granted and admin-consented.

In [ ]:
print("Authenticating with Microsoft Graph...")

token_url = f"https://login.microsoftonline.com/{TENANT_ID}/oauth2/v2.0/token"

token_response = requests.post(token_url, data={
    "client_id":     CLIENT_ID,
    "client_secret": CLIENT_SECRET,
    "scope":         "https://graph.microsoft.com/.default",
    "grant_type":    "client_credentials"
})
token_response.raise_for_status()

access_token = token_response.json()["access_token"]
headers = {
    "Authorization": f"Bearer {access_token}",
    "Content-Type":  "application/octet-stream"
}

print("Authentication successful.")

## Step 4: Upload to SharePoint

Files are uploaded using the Graph API [upload small file](https://learn.microsoft.com/en-us/graph/api/driveitem-put-content)
endpoint (`PUT /drives/{drive-id}/root:/{path}:/content`). This method is suitable
for files up to ~4 MB. For larger files, the resumable upload session API should
be used instead.

Both the timestamped archive and the LATEST snapshot are uploaded.

In [ ]:
def upload_to_sharepoint(local_file: str, remote_filename: str) -> None:
    """Upload a local file to the configured SharePoint drive folder."""
    upload_url = (
        f"https://graph.microsoft.com/v1.0/drives/{DRIVE_ID}"
        f"/root:/{FOLDER_PATH}/{remote_filename}:/content"
    )
    with open(local_file, "rb") as f:
        response = requests.put(upload_url, headers=headers, data=f)
    response.raise_for_status()
    print(f"Uploaded: {remote_filename}")


print("Uploading files to SharePoint...")
upload_to_sharepoint(local_path,  filename)
upload_to_sharepoint(latest_path, latest_filename)
print("All uploads successful.")

## Summary

This notebook performed the following steps:
1. Read `cx_nps_rolling_5w` from the Lakehouse Delta table
2. Validated the row count (fails fast on empty table)
3. Exported two CSVs to the local Lakehouse staging folder
4. Authenticated to Microsoft Graph via client credentials flow
5. Uploaded both CSV files to SharePoint

---

### Adapting to Your Workspace

- [ ] Set all environment variables listed in `.env.example`
- [ ] Attach this notebook to the correct Lakehouse in your Fabric workspace
- [ ] Register an Azure AD app with `Files.ReadWrite.All` and admin consent
- [ ] Update `TABLE_NAME` if your Dataflow writes to a different table name
- [ ] Update `FOLDER_PATH` to your target SharePoint folder
- [ ] For files > 4 MB, replace the PUT upload with a resumable upload session